# Pycaret (Baseline)

## Biblioteca / Configuração

In [1]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [2]:
# Acesso aos modulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação dos dados
import pandas as pd
import numpy as np
import pickle

import json
from datetime import datetime
import os

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from config.paths import *
from config.function_models_pycaret import *

# Modelos / Machine Learning
from pycaret.classification import *
#from pycaret.classification import get_config, finalize_model, predict_model

# Métricas e validação
from sklearn.metrics import (precision_score, recall_score, f1_score,  roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('Ambiente Configurado')

Diretórios carregadas com sucesso
Funções pycaret carregadas com sucesso
Ambiente Configurado


## Parâmetros Globais

In [3]:
# define a coluna alvo do modelo
TARGET = 'FPD'
# garante reprodutibilidade dos experimentos
RANDOM_STATE = 42
# Definindo o número de folds na validação cruzada
CV = 5
# quantidade de modelos selecionadas
n_select = 7
# data de execução do notebook (para versionamento/controle)
DATA_EXECUCAO = datetime.now().strftime('%d-%m-%Y')
# versão do pipeline/modelo
VERSAO = 'V1 - baseline(picaret)'

## Carregamento dos dados 

In [4]:
# Carregar datasets processados
train = pd.read_parquet(PROCESSED_DIR / 'abt01_train.parquet')
test =  pd.read_parquet(PREDICTIONS_DIR / 'abt01_test.parquet')

print(f'Treino: {train.shape}')
print(f'Teste: {test.shape}')

Treino: (831081, 119)
Teste: (389550, 19)


## Artefatos

In [5]:
# Carregar a lista de features do features_stage_02.pkl
artifact_path = Path(ARTIFACT_DIR) / 'features_stage_02.pkl'

with open(ARTIFACT_DIR / 'features_stage_02.pkl', 'rb') as f:
    features_stage_02 = pickle.load(f)

### Seleção da importância das variáveis

In [6]:
# Backup dos dados originais
train_01 = train.copy()

# Carregar lista de features do Feature selection e recolocar a target original
train_01 = train[features_stage_02 + [TARGET]].copy()

print(f'Treino: {train_01.shape}')

Treino: (831081, 19)


In [7]:
train_01.head()

,SCORE_02,var_73,var_35,SCORE_RATEO,BOLSA_FAMILIA,var_41,var_34,AUX_EMRG,IDADE,FUNC_PRIVADO,ADMITIDO,REGIAO_POSTAL,var_62,var_68,var_03,var_05,var_40,APOSENTADO,FPD
0,636.0,0.0,23.01,1.131673,0,61.91,27.40,0,40,1,1,4,94.93,96.95,33,2,27.78,0,0
1,518.0,0.0,304.00,0.948718,0,1.05,304.00,0,43,1,1,6,2.50,0.00,14,2,0.19,0,1
2,750.0,0.0,99.96,1.207729,0,100.00,100.00,0,42,1,1,7,99.77,70.12,33,4,99.96,0,0
3,679.0,0.0,58.11,1.114943,0,99.70,66.23,1,36,0,0,6,80.93,99.93,50,1,85.75,0,1
4,722.0,1000.0,45.12,1.162641,0,96.51,59.19,0,40,1,1,7,98.01,72.63,33,2,72.93,0,0


## Setup do PyCaret

In [8]:
# Necessário para evitar conflito de índices duplicados no PyCaret
df_train = train_01.reset_index(drop=True)
df_test = test.reset_index(drop=True)

# Setup completo para modelagem de CLASSIFICAÇÃO no PyCaret
exp_clf101 = setup(
    data=df_train,                        # DataFrame: Dataset que você deseja usar para modelagem.
    target= TARGET,                       # str: Nome da coluna target (DEVE ser categórica).
    #train_size=0.7,                       # float: Proporção do dataset para treinamento.
    test_data=df_test,                    # DataFrame opcional para teste externo.
    index=False,                          # força PyCaret a usar RangeIndex
    ordinal_features=None,                # dict: Mapeamento de colunas ordinais e sua respectiva ordem.
    numeric_features=None,                # list: Lista de colunas tratadas como numéricas.
    categorical_features=None,            # list: Lista de colunas tratadas como categóricas.
    date_features=None,                   # list: Lista de colunas de data.
    text_features=None,                   # list: Lista de colunas de texto.
    ignore_features=None,                 # list: Lista de colunas a serem ignoradas.
    preprocess=False,                     # bool: Se True, aplica todo o pipeline de preprocessamento.
    imputation_type='simple',             # str: Tipo de imputação ('simple' ou 'iterative').
    numeric_imputation='mean',            # str: Imputação para colunas numéricas ('mean' ou 'median').
    categorical_imputation='mode',        # str: Imputação para colunas categóricas ('mode' ou 'constant').
    normalize=False,                      # bool: Se True, normaliza as colunas numéricas.
    normalize_method='zscore',            # str: Método de normalização ('zscore', 'minmax', etc.).
    transformation=False,                 # bool: Se True, transforma colunas para aproximar a normalidade.
    transformation_method='yeo-johnson',  # str: Método de transformação ('yeo-johnson' ou 'box-cox').
    pca=False,                            # bool: Se True, aplica PCA para redução de dimensionalidade.
    pca_method='linear',                  # str: Método de PCA ('linear', 'kernel', 'incremental').
    pca_components=None,                  # int ou float: Número de componentes ou variância explicada.
    bin_numeric_features=None,            # list: Lista de colunas numéricas para binning.
    remove_outliers=False,                # bool: Se True, remove outliers automaticamente.
    outliers_threshold=0.05,              # float: Proporção de outliers permitida.
    remove_multicollinearity=True,        # bool: Se True, remove features multicolineares.
    multicollinearity_threshold=0.9,      # float: Limite para correlação entre variáveis.
    polynomial_features=False,            # bool: Se True, cria features polinomiais.
    polynomial_degree=2,                  # int: Grau das features polinomiais.
    feature_selection=False,              # bool: Se True, aplica seleção automática de features.
    n_features_to_select=0.6,             # float ou int: Proporção ou número de features mantidas.
    fix_imbalance=False,                  # bool: Se True, aplica balanceamento automático de classes.
    fix_imbalance_method=None,            # objeto: Método de balanceamento (default = SMOTE).
    fold=CV,                              # int: Número de folds para validação cruzada.
    fold_shuffle=True,                    # bool: Se True, embaralha os dados nos folds.
    data_split_shuffle=True,              # bool: Se True, embaralha os dados antes do split.
    n_jobs=-1,                            # int: Número de cores utilizados (-1 usa todos).
    session_id=RANDOM_STATE,              # int: Seed para reprodutibilidade.
    verbose=False,                        # bool: Se True, imprime logs detalhados.
    profile=False                         # bool: Se True, gera um profile report do dataset.
)

## Algoritmos de aprendizado para classificação


In [9]:
# Lista de modelos de classificação do PyCaret
model_ids = [
    'lr',        # Logistic Regression
    #'knn',      # K Neighbors Classifier
    #'nb',       # Naive Bayes
    'dt',        # Decision Tree Classifier
    #'svm',      # Support Vector Machine
    #'rbfsvm',   # SVM (RBF Kernel)
    #'gpc',      # Gaussian Process Classifier
    #'mlp',      # Multi Layer Perceptron
    #'ridge',    # Ridge Classifier
    #'qda',      # Quadratic Discriminant Analysis
    'rf',        # Random Forest
    'et',        # Extra Trees Classifier
    #'ada',      # AdaBoost Classifier
    'gbc',       # Gradient Boosting Classifier
    'xgboost',   # Extreme Gradient Boosting
    'lightgbm',  # Light Gradient Boosting Machine
    #'catboost'   # CatBoost Classifier
]

## Comparação de Modelos

In [10]:
# Comparar modelos de classificação e selecionar top 3 pelo AUC (foco em separar bons/mau pagadores)
print('\n🤖 Comparando modelos...')

compared_models = compare_models(
    include=model_ids,             # Lista de modelos de classificação
    # exclude=None,                # Modelos a excluir
    fold=CV,                       # Número de dobras da validação cruzada
    round=3,                       # Casas decimais nas métricas
    cross_validation=True,         # Executa validação cruzada
    sort='AUC',                    # Métrica para ordenação dos modelos
    n_select=n_select,             # Quantidade de melhores modelos retornados
    # budget_time=None,            # Tempo máximo (em minutos) para a comparação
    turbo=True,                    # Exclui modelos mais lentos automaticamente
    errors='raise',                # Ignora falhas durante o treinamento
    # fit_kwargs=None,             # Argumentos extras passados ao método fit
    # groups=None,                 # Grupos para validação cruzada estratificada
    # experiment_custom_tags=None, # Tags customizadas do experimento
    # engine=None,                 # Backend / engine do modelo
    verbose=True,                  # Exibe progresso da execução
    # parallel=None                # Backend de paralelização
)

print('\n✅ Comparação de modelos concluída!')
print(f'\n📊 Top modelos selecionados')


🤖 Comparando modelos...


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.772,0.707,0.081,0.557,0.141,0.088,0.144,5.508
gbc,Gradient Boosting Classifier,0.772,0.706,0.075,0.561,0.133,0.082,0.140,67.770
xgboost,Extreme Gradient Boosting,0.772,0.705,0.092,0.546,0.158,0.097,0.150,4.496
lr,Logistic Regression,0.769,0.695,0.039,0.533,0.073,0.043,0.093,30.776
rf,Random Forest Classifier,0.767,0.685,0.109,0.493,0.179,0.104,0.144,74.656
et,Extra Trees Classifier,0.763,0.668,0.114,0.454,0.182,0.098,0.131,116.418
dt,Decision Tree Classifier,0.668,0.548,0.325,0.300,0.312,0.093,0.093,5.924



✅ Comparação de modelos concluída!

📊 Top modelos selecionados


In [11]:
# Gerar tabela final comparavel por algoritmo

# pega splits internos do PyCaret
X_train = get_config('X_train')
y_train = get_config('y_train')
X_test  = get_config('X_test')
y_test  = get_config('y_test')

models_list = compared_models if isinstance(compared_models, list) else [compared_models]

# Tabela consolidada de metricas
tabela_metricas = build_metrics_table(
    models_list,
    X_train, y_train,
    X_test, y_test,
    finalize_model
)

tabela_metricas

,Algoritmo,Conjunto,Acuracia,Precisao,Recall,AUC_ROC,GINI,KS
0,DecisionTreeClassifier,Treino,0.999628,0.998683,0.999715,1.000000,0.999999,0.999410
1,DecisionTreeClassifier,Teste,0.999802,0.999388,0.999739,1.000000,1.000000,0.999652
2,ExtraTreesClassifier,Treino,0.999628,0.998683,0.999715,1.000000,0.999999,0.999410
3,ExtraTreesClassifier,Teste,0.999802,0.999388,0.999739,1.000000,1.000000,0.999652
4,GradientBoostingClassifier,Treino,0.772203,0.566931,0.073466,0.706894,0.413788,0.301111
5,GradientBoostingClassifier,Teste,0.776201,0.559383,0.055945,0.698496,0.396993,0.289878
6,LGBMClassifier,Treino,0.772826,0.569370,0.082223,0.710869,0.421739,0.306380
7,LGBMClassifier,Teste,0.776760,0.564186,0.063063,0.703140,0.406281,0.296320
8,LogisticRegression,Treino,0.768630,0.530380,0.016900,0.691610,0.383219,0.281389
9,LogisticRegression,Teste,0.773408,0.493377,0.016888,0.680571,0.361143,0.265604


In [12]:
# Selecionar melhor modelo com maior KS no Teste e menor overfitting

# Cria visao Treino vs Teste
pivot = tabela_metricas.pivot(index='Algoritmo', columns='Conjunto', values='KS')
pivot['gap'] = abs(pivot['Treino'] - pivot['Teste'])

# Remove modelos suspeitos (KS perfeito demais no teste)
pivot_validos = pivot[pivot['Teste'] < 0.98]

# Escolhe campeão: maior KS Teste e menor gap
melhor_algoritmo = (
    pivot_validos
    .sort_values(['Teste','gap'], ascending=[False, True])
    .index[0]
)

# Recupera objeto do modelo
models_list = compared_models if isinstance(compared_models, list) else [compared_models]
best_model = next(m for m in models_list if type(m).__name__ == melhor_algoritmo)

best_model

XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cpu', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=-1, num_parallel_tree=None, ...)

In [13]:
# Pegar o melhor modelo e finalizar modelo campeao
final_best_model = finalize_model(best_model)

## Salvamento de Resultados

### Salvar o melhor modelo

In [14]:
# Salvar o melhor modelo em arquivo para reutilização
with open(MODELS_DIR / 'pycaret_model.pkl', 'wb') as f:
    pickle.dump(final_best_model, f)

### Métricas de MODELO

In [15]:
# pegar conjuntos internos do PyCaret (holdout do experimento)
X_val = exp_clf101.get_config('X_test')
y_val = exp_clf101.get_config('y_test')

In [16]:
# Predições do melhor modelo
y_pred = final_best_model.predict(X_val)
y_pred_proba = final_best_model.predict_proba(X_val)[:, 1]  # prob da classe positiva

# confusion matrix
tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

# AUC, GINI, KS
auc = roc_auc_score(y_val, y_pred_proba)
gini = 2 * auc - 1
fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
ks = np.max(tpr - fpr)

In [17]:
# dataframe final
metrics_row = {
    'Precision': precision_score(y_val, y_pred, zero_division=0),
    'Recall': recall_score(y_val, y_pred, zero_division=0),
    'F1_score': f1_score(y_val, y_pred, zero_division=0),
    'AUC': auc,
    'GINI': gini,
    'KS': ks,
    'TP': tp,
    'TN': tn,
    'FP': fp,
    'FN': fn,
    'Versao_Modelo': VERSAO,
    'Data_Execucao': DATA_EXECUCAO,
}

df_metrics = pd.DataFrame([metrics_row])
df_metrics

,Precision,Recall,F1_score,AUC,GINI,KS,TP,TN,FP,FN,Versao_Modelo,Data_Execucao
0,0.589163,0.07493,0.132951,0.712737,0.425474,0.308351,6611,296711,4610,81618,V1 - baseline(picaret),27-02-2026


In [18]:
# salvar metricas versionadas do modelo
path_metrics = METRICS_DIR / 'model_metrics_treino.csv'

if os.path.exists(path_metrics):
    df_hist = pd.read_csv(path_metrics)

    # se versao ja existe, atualiza
    if VERSAO in df_hist['Versao_Modelo'].values:
        df_hist.loc[
            df_hist['Versao_Modelo'] == VERSAO,
            df_metrics.columns
        ] = df_metrics.iloc[0].values
    else:
        df_hist = pd.concat([df_hist, df_metrics], ignore_index=True)

else:
    df_hist = df_metrics

df_hist.to_csv(path_metrics, index=False)

In [19]:
# Validação
path_metrics = METRICS_DIR / 'model_metrics_treino.csv'
df_metrics = pd.read_csv(path_metrics)

# ordenar da mais recente pra mais antiga
df_metrics = df_metrics.sort_values('Data_Execucao', ascending=False)

df_metrics.head()

,Precision,Recall,F1_score,AUC,GINI,KS,TP,TN,FP,FN,Versao_Modelo,Data_Execucao
0,0.589163,0.07493,0.132951,0.712737,0.425474,0.308351,6611,296711,4610,81618,V1 - baseline(picaret),27-02-2026
